# 통합 실행용 커널 서버 - Qwen 4-bit + 위치 압축 커널, LoRA 제외

- 최종 A100 커널 실험의 INT4/SNQK 위치 압축 추론 경로를 서버 생성 모델에 사용합니다.
- Qwen3.8-27B 4-bit 베이스만 로드하며 LoRA 어댑터는 로드하지 않습니다.
- 기존 앱 기능을 유지합니다: 계획 기반 8단계 동화, 이미지, 선택 해설, 캐릭터 채팅, Whisper 음성인식, XTTS 음성합성.
- LLM이 감정 점수를 만들지 않으며, 동화와 선택 기록을 바탕으로 문장 해설을 제공합니다.
- 실행은 A100 40GB 런타임을 사용합니다. Drive 마운트 없이 Hugging Face 캐시를 Colab 임시 디스크에 둡니다.


In [ ]:
# 셀 1: 통합 서버 의존성 설치
import os
import shutil
import subprocess
import sys
from collections import deque
from pathlib import Path

SETUP_LOG = Path("/content/fairytale_setup.log")
SETUP_LOG.write_text(f"Python {sys.version}\n", encoding="utf-8")

def run_setup(command, stage, env=None):
    print(f"\n[{stage}]", flush=True)
    tail = deque(maxlen=60)
    with SETUP_LOG.open("a", encoding="utf-8") as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding="utf-8", errors="replace", env=env) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
                tail.append(line)
            status = process.wait()
    if status:
        raise RuntimeError(f"{stage} 실패 (종료 코드 {status})\n{''.join(tail)[-6000:]}\n로그: {SETUP_LOG}")

def pip_install(python, packages, stage, options=(), env=None):
    run_setup([python, "-m", "pip", "install", "--disable-pip-version-check",
               "--upgrade", *options, *packages], stage, env=env)

# 최종 커널 실험과 동일한 Unsloth/Transformers/FLA 버전을 사용합니다.
MAIN_PACKAGES = [
    "huggingface-hub>=1.5.0,<2.0", "transformers==5.17.0",
    "unsloth==2026.10.5", "unsloth_zoo==2026.10.5", "peft==0.21.1",
    "bitsandbytes==0.50.2", "triton==3.6.0", "flash-linear-attention==0.4.2",
    "ninja", "diffusers==0.36.0", "accelerate", "safetensors", "sentencepiece",
    "fastapi==0.141.1", "uvicorn[standard]", "python-multipart", "nest-asyncio",
    "requests==2.32.4", "pillow==11.3.0", "virtualenv", "faster-whisper>=1.1.0",
    "invisible_watermark", "diskcache",
]
pip_install(sys.executable, MAIN_PACKAGES, "Qwen 커널·삽화·음성 인식 의존성",
            options=["--only-binary=tokenizers,sentencepiece"])
# Colab ships Pillow's native extension in its base image. Reinstall the same
# version to avoid mixing Python files with a different prebuilt extension.
run_setup([sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-cache-dir",
           "pillow==11.3.0"], "Pillow 바이너리 일치")
# The custom attention path does not use xformers or TorchAO. Colab's prebuilt
# versions conflict with this runtime's Torch and make Diffusers import fail.
for optional_package in ["xformers", "torchao"]:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", optional_package],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
run_setup([sys.executable, "-c",
           "import unsloth; import torch, transformers, tokenizers, fastapi, faster_whisper; "
           "from unsloth import FastModel; from fla.ops.gated_delta_rule import fused_recurrent_gated_delta_rule; "
           "from diffusers import StableDiffusionPipeline; "
           "print('메인 import OK', 'torch='+torch.__version__, 'transformers='+transformers.__version__, "
           "'tokenizers='+tokenizers.__version__)"] , "메인 환경 import 점검")

# XTTS는 Transformers 충돌을 피하도록 기존처럼 별도 환경에서 구동합니다.
XTTS_VENV = "/content/test_v1_xtts_env"
XTTS_PYTHON = f"{XTTS_VENV}/bin/python"
if not os.path.isfile(XTTS_PYTHON):
    run_setup([sys.executable, "-m", "virtualenv", "--system-site-packages", XTTS_VENV], "XTTS 환경 생성")
pip_install(XTTS_PYTHON, ["pip"], "XTTS pip 준비")
pip_install(XTTS_PYTHON,
            ["coqui-tts==0.27.5", "transformers==4.57.6", "tokenizers==0.22.2",
             "huggingface-hub==0.36.0", "fastapi", "uvicorn[standard]", "python-multipart"],
            "XTTS 의존성", options=["--only-binary=tokenizers"])
run_setup([XTTS_PYTHON, "-c", "import torch, transformers; from TTS.api import TTS; "
           "print('XTTS import OK', 'transformers='+transformers.__version__)"], "XTTS 환경 import 점검")

if not shutil.which("cloudflared"):
    cloudflared_deb = "/content/cloudflared.deb"
    run_setup(["wget", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
               "-O", cloudflared_deb], "Cloudflare 다운로드")
    run_setup(["dpkg", "-i", cloudflared_deb], "Cloudflare 설치")

print("\n설치/import 점검 완료. 런타임을 재시작한 뒤 셀 2부터 실행하세요.")


In [ ]:
from pathlib import Path
import base64, zipfile, io, hashlib, os, sys, json, time, traceback, collections, re
r=Path('/content/fairytale_repair');r.mkdir(exist_ok=True);os.chdir(r);(r/'results').mkdir(exist_ok=True)
source_blob=base64.b64decode('...')
assert hashlib.sha256(source_blob).hexdigest()=='c5a8821599e6fea0057fa980ad38afe5c4cbc8a0043e11b12aebada83ea19c7b'
with zipfile.ZipFile(io.BytesIO(source_blob)) as z:z.extractall(r)
sys.path.insert(0,str(r/'research_src'))


In [ ]:
# 셀 2: A100용 Qwen 4-bit 베이스와 기존 보조 모델 설정 (LoRA 미사용)
import gc
import json
import os
import torch
from pathlib import Path
from huggingface_hub import login, snapshot_download
from unsloth import FastModel

HF_CACHE_ROOT = "/content/fairytale_hf_cache"
os.environ["HF_HOME"] = f"{HF_CACHE_ROOT}/home"
os.environ["HF_HUB_CACHE"] = f"{HF_CACHE_ROOT}/hub"
os.environ["TTS_HOME"] = "/content/fairytale_xtts_cache"
STT_CACHE_ROOT = "/content/fairytale_stt_cache"
STT_MODEL_NAME = "large-v3-turbo"
for directory in [os.environ["HF_HOME"], os.environ["HF_HUB_CACHE"], os.environ["TTS_HOME"], STT_CACHE_ROOT]:
    Path(directory).mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

XTTS_PYTHON = "/content/test_v1_xtts_env/bin/python"
BASE_ID = "unsloth/Qwen3.8-27B-unsloth-bnb-4bit"
BASE_REVISION = "8aa5f05d26b7205477066e1449e0af13f762a299"
MAX_SEQ_LENGTH = 32768
MAX_NEW_TOKENS = 4600
XTTS_ACCEPT_LICENSE = True
OBSERVATION_SOURCE = "qwen_behavior_observation_v1"

if not torch.cuda.is_available():
    raise RuntimeError("A100 GPU 런타임이 필요합니다.")
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
GPU_NAME = torch.cuda.get_device_name(0)
if torch.cuda.get_device_capability() != (8, 0):
    raise RuntimeError(f"최종 커널은 A100 SM80에서 검증했습니다. 현재 GPU: {GPU_NAME}")
if GPU_VRAM_GB < 36:
    raise RuntimeError(f"A100 40GB가 필요합니다. 현재 VRAM: {GPU_VRAM_GB:.1f}GB")

HIGH_VRAM_MODE = True
IMAGE_DEVICE = "cuda"
XTTS_DEVICE = "cuda"
STT_DEVICE = "cuda"
STT_COMPUTE_TYPE = "float16"
os.environ["XTTS_ACCEPT_LICENSE"] = "true"
os.environ["COQUI_TOS_AGREED"] = "1"
os.environ["XTTS_DEVICE"] = XTTS_DEVICE

BASE_DIR = Path(HF_CACHE_ROOT) / "qwen38_27b_kernel_base"
if not (BASE_DIR / "config.json").is_file():
    snapshot_download(BASE_ID, revision=BASE_REVISION, local_dir=str(BASE_DIR))

print(f"GPU: {GPU_NAME} / VRAM {GPU_VRAM_GB:.1f}GB")
print("Qwen 4-bit 베이스 모델 로딩 중 (LoRA를 불러오지 않습니다)...")
model, tokenizer = FastModel.from_pretrained(
    model_name=str(BASE_DIR), max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True,
    full_finetuning=False, dtype=torch.float16, text_only=True,
    local_files_only=True, offload_embedding=False,
)
if not getattr(model.config, "architectures", None):
    model.config.architectures = ["Qwen3_5ForCausalLM"]
FastModel.for_inference(model)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

exec((r / "helpers.py").read_text(encoding="utf-8"), globals())
from repair_runtime import install_fla_decode, AttentionDispatch
import oss_graphs27
install_fla_decode(model)
oss_graphs27.install(model)
oss_graphs27.install_linear(model)
dispatch = AttentionDispatch(model)

print("✅ Qwen 4-bit + 최종 INT4/SNQK 위치 압축 커널 준비 완료 (LoRA 미적용)")
print("삽화: DreamShaper 8 / 음성인식: faster-whisper large-v3-turbo / 낭독: XTTS-v2")


In [ ]:
# 셀 3: Qwen 호출, 느슨한 JSON 파싱, 텍스트 정리 유틸리티
import ast
import html
import hashlib
import math
import re
import unicodedata
import threading


LLM_LOCK = threading.Lock()

def qwen_prompt(user_prompt):
    return chat_text(user_prompt.strip())


def call_llm(prompt, max_tokens, temperature=0.55, top_p=0.9):
    prompt_text = qwen_prompt(prompt)
    encoded = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).to(model.device)
    prompt_tokens = int(encoded.input_ids.shape[1])
    max_new = max(1, min(int(max_tokens), MAX_NEW_TOKENS, MAX_SEQ_LENGTH - prompt_tokens))
    if prompt_tokens >= MAX_SEQ_LENGTH:
        raise RuntimeError(f"LLM 입력이 컨텍스트 한도({MAX_SEQ_LENGTH})를 넘었습니다.")
    with LLM_LOCK:
        cache = dispatch.cache("position", prompt_tokens, max_new)
        generation_config = copy.deepcopy(model.generation_config)
        generation_config.cache_implementation = None
        generation_config.max_length = prompt_tokens + max_new
        generate_fn = getattr(model, "_old_generate", model.generate)
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
            result = generate_fn(
                **encoded, max_new_tokens=max_new, do_sample=temperature > 0,
                temperature=max(temperature, 1e-5), top_p=top_p,
                repetition_penalty=1.08, pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id, use_cache=True,
                return_dict_in_generate=True, past_key_values=cache,
                generation_config=generation_config,
            )
        generated_ids = result.sequences[0, prompt_tokens:].tolist()
        return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def clean_model_text(value):
    text = html.unescape(str(value or ""))
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    text = re.sub(r"```(?:json)?", "", text, flags=re.I)
    text = text.replace("</CHOICES>", "").replace("</STORY>", "")
    text = re.sub(r"(?m)^\s*(?:Human|Assistant)\s*:\s*", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def strip_json_comments(text):
    output, index, quoted, escaped = [], 0, False, False
    while index < len(text):
        char = text[index]
        nxt = text[index + 1] if index + 1 < len(text) else ""
        if quoted:
            output.append(char)
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == '"':
                quoted = False
            index += 1
        elif char == '"':
            quoted = True
            output.append(char)
            index += 1
        elif char == "/" and nxt == "/":
            newline = text.find("\n", index)
            index = len(text) if newline < 0 else newline + 1
        elif char == "/" and nxt == "*":
            close = text.find("*/", index + 2)
            index = len(text) if close < 0 else close + 2
        else:
            output.append(char)
            index += 1
    return "".join(output)


def parse_json_loose(fragment):
    if not isinstance(fragment, str):
        return fragment if isinstance(fragment, (dict, list)) else None
    repaired = strip_json_comments(fragment)
    repaired = repaired.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
    repaired = re.sub(r",\s*([}\]])", r"\1", repaired)
    for candidate in (fragment, repaired):
        try:
            return json.loads(candidate)
        except Exception:
            try:
                value = ast.literal_eval(candidate)
                if isinstance(value, (dict, list)):
                    return value
            except Exception:
                pass
    return None


def extract_json_loose(text):
    # 설명문 안의 대괄호보다 실제 JSON 객체/배열을 우선 안전하게 찾습니다.
    text = clean_model_text(text)
    pairs = {"{": "}", "[": "]"}
    for start, opener in enumerate(text):
        if opener not in pairs:
            continue
        stack, quoted, escaped = [pairs[opener]], False, False
        for position in range(start + 1, len(text)):
            char = text[position]
            if quoted:
                if escaped:
                    escaped = False
                elif char == "\\":
                    escaped = True
                elif char == '"':
                    quoted = False
                continue
            if char == '"':
                quoted = True
            elif char in pairs:
                stack.append(pairs[char])
            elif char in "}]":
                if not stack or char != stack[-1]:
                    break
                stack.pop()
                if not stack:
                    parsed = parse_json_loose(text[start:position + 1])
                    if isinstance(parsed, (dict, list)):
                        return parsed
                    break
    return None


def sentences(text):
    return [part.strip() for part in re.split(r"(?<=[.!?…])\s+|\n+", clean_model_text(text)) if part.strip()]


def clean_story(text, headings=()):
    text = clean_model_text(text)
    text = re.sub(r"(?m)^\s*(?:제목|장면|본문|단계)\s*:\s*[^\n]*$", "", text)
    text = re.sub(r"(?m)^\s*#{1,6}\s+.*$", "", text)
    text = text.replace("**", "").replace("__", "")
    text = "".join(char for char in text if unicodedata.category(char) not in {"So", "Sk"})
    normalized_headings = {
        re.sub(r"\s+", "", clean_model_text(heading))
        for heading in headings if clean_model_text(heading)
    }
    kept_lines = []
    for line in text.splitlines():
        compact = re.sub(r"\s+", "", line.strip())
        if compact and compact in normalized_headings:
            continue
        kept_lines.append(line)
    text = "\n".join(kept_lines)
    text = re.sub(r"\s+([,.!?])", r"\1", text)
    text = re.sub(r"([!?])\1+", r"\1", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def compact_json(data, limit=5200):
    # 필드와 배열 구조는 유지하고 긴 문자열만 줄입니다.
    value = json.loads(json.dumps(data, ensure_ascii=False))
    for _ in range(12):
        result = json.dumps(value, ensure_ascii=False, separators=(",", ":"))
        if len(result) <= limit:
            return result
        strings = []
        def collect(node):
            if isinstance(node, dict):
                for key, item in node.items():
                    if isinstance(item, str) and len(item) > 32:
                        strings.append((node, key, item))
                    else:
                        collect(item)
            elif isinstance(node, list):
                for key, item in enumerate(node):
                    if isinstance(item, str) and len(item) > 32:
                        strings.append((node, key, item))
                    else:
                        collect(item)
        collect(value)
        if not strings:
            return result
        parent, key, text = max(strings, key=lambda item: len(item[2]))
        parent[key] = text[:max(32, len(text) - max(64, len(result) - limit))].rstrip() + "…"
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"))


def fit_generation_prompt(prompt, sections, max_tokens):
    runtime = globals().get("llm")
    tokenize = getattr(runtime, "tokenize", None)
    if not callable(tokenize):
        return prompt
    context_size = globals().get("N_CTX", 8192)
    budget = max(256, context_size - max_tokens - 96)
    entries = []
    for title, value, limit, minimum in sections:
        block = compact_json(value, limit) if isinstance(value, (dict, list)) else str(value)
        entries.append({"title": title, "value": value, "limit": limit,
                        "minimum": minimum, "block": block})
    for _ in range(32):
        tokens = tokenize(qwen_prompt(prompt).encode("utf-8"), add_bos=True)
        if len(tokens) <= budget:
            return prompt
        candidates = [item for item in entries if item["limit"] > item["minimum"]]
        if not candidates:
            break
        item = max(candidates, key=lambda entry: len(entry["block"]))
        item["limit"] = max(item["minimum"], int(item["limit"] * 0.72))
        value = item["value"]
        if isinstance(value, (dict, list)):
            smaller = compact_json(value, item["limit"])
        else:
            # 이전 장면의 첫 행동과 끝의 갈림점을 모두 남깁니다.
            half = max(1, (item["limit"] - 3) // 2)
            smaller = str(value)[:half] + " … " + str(value)[-half:]
        marker = f"[{item['title']}]\n"
        # 문자열 전체가 아니라 해당 제목 아래의 자료만 바꿉니다.
        prompt = prompt.replace(marker + item["block"], marker + smaller, 1)
        item["block"] = smaller
    return prompt


# 점수 없이 실제 사건과 선택 행동만 기록합니다.
def request_observation_json(prompt, max_tokens):
    try:
        raw = call_llm(prompt, max_tokens=max_tokens, temperature=0.15, top_p=0.8)
        data = extract_json_loose(raw)
        return data if isinstance(data, dict) else {}
    except Exception as error:
        print(f"[정보] 관찰 기록을 만들지 못했습니다. 이야기는 계속 진행합니다: {type(error).__name__}")
        return {}


def behavior_observation(data):
    data = data if isinstance(data, dict) else {}
    tags = data.get("behavior_tags", [])
    tags = list(dict.fromkeys(
        clean_model_text(tag)[:24] for tag in tags
        if isinstance(tag, str) and clean_model_text(tag)
    ))[:4] if isinstance(tags, list) else []
    return {"source": OBSERVATION_SOURCE, "behavior_tags": tags}


def discard_emotion_scores(game):
    # 이전 runtime_state도 사건 인용과 행동 기록만 복구합니다.
    state = game["state"]
    for scene in state.get("scenes", []):
        old = scene.pop("emotion", {})
        if isinstance(old, dict) and not isinstance(scene.get("memory"), dict):
            scene["memory"] = {
                "scene_facts": source_quotes(old.get("scene_facts"), scene["text"]),
                "open_threads": source_quotes(old.get("open_threads"), scene["text"], 2),
            }
    choices = list(state.get("choices", []))
    for decision in state.get("history", []):
        choices.extend(decision.get("offered_choices", []))
        if isinstance(decision.get("choice"), dict):
            choices.append(decision["choice"])
    for choice in choices:
        old = choice.pop("emotion", {})
        observation = choice.get("observation")
        if isinstance(observation, dict):
            choice["observation"] = behavior_observation(observation)
        elif isinstance(old, dict) and old.get("behavior_tags"):
            choice["observation"] = behavior_observation(old)
    game.setdefault("settings", {})["emotion_scoring_enabled"] = False
    return game


def source_quotes(items, source, maximum=4):
    if not isinstance(items, list):
        return []
    source_key = re.sub(r"\s+", "", source)
    quotes = []
    for item in items:
        text = clean_model_text(item)
        if text and re.sub(r"\s+", "", text) in source_key and text not in quotes:
            quotes.append(text)
        if len(quotes) == maximum:
            break
    return quotes


def extract_scene_memory(story_text, game=None):
    context = analysis_context(game) if game is not None else {}
    prompt = f'''너는 이미 쓰인 동화에서 실제 사건만 기록한다.
[본문]
{story_text}
[인물과 현재 단계]
{json.dumps(context, ensure_ascii=False)}
규칙:
- scene_facts에는 실제 행동·발견·물건의 획득/분실·약속·관계 변화를 최대 4개 골라 본문 문장 그대로 적는다. 각 인용은 160자 이내다.
- open_threads에는 아직 풀리지 않은 문제나 물음을 본문에서 최대 2개 그대로 인용한다.
- 계획의 목표, 앞으로 할 일, 인물이 상상한 결과를 이미 달성한 사건으로 기록하지 않는다.
- 근거가 없으면 빈 배열이다. 감정 라벨, 감정 점수, 성향 수치, 확률을 생성하지 않는다.
JSON만 출력한다.
{{"scene_facts":["실제 사건 문장"],"open_threads":["미해결 물음 문장"]}}'''
    data = request_observation_json(prompt, 650)
    return {
        "scene_facts": source_quotes(data.get("scene_facts"), story_text),
        "open_threads": source_quotes(data.get("open_threads"), story_text, 2),
    }


# ── DreamShaper 8 삽화: Qwen VRAM을 위해 필요할 때만 로드합니다. ────────────
image_pipe = None


def get_image_pipe():
    global image_pipe
    if image_pipe is not None:
        return image_pipe
    from diffusers import DPMSolverMultistepScheduler, StableDiffusionPipeline
    dtype = torch.float16 if IMAGE_DEVICE == "cuda" else torch.float32
    print(f"DreamShaper 8 로딩 중... ({IMAGE_DEVICE})")
    pipe = StableDiffusionPipeline.from_pretrained(
        "Lykon/dreamshaper-8",
        torch_dtype=dtype,
        safety_checker=None,
        requires_safety_checker=False,
    )
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, algorithm_type="dpmsolver++")
    image_pipe = pipe.to(IMAGE_DEVICE)
    image_pipe.enable_attention_slicing()
    return image_pipe


def generate_illustration(story_text, genre="판타지"):
    style = {
        "판타지": "magical Korean children picture book, soft watercolors, glowing forest",
        "모험": "Korean children picture book, adventurous landscape, warm sunlight",
        "우정": "Korean children picture book, warm friendship, gentle colors",
    }.get(genre, "Korean children picture book, warm detailed illustration")
    visual = call_llm(
        "다음 동화 장면을 삽화 프롬프트용 영어 한 문장으로 바꿔라. 인물, 장소, 행동만 쓰고 22단어 이내로 쓴다.\n\n" + story_text[-1100:],
        max_tokens=70,
        temperature=0.2,
    )
    visual = re.sub(r"[가-힣]+", "", visual).strip() or "a child follows a glowing path through an enchanted forest"
    result = get_image_pipe()(
        prompt=f"{visual}, {style}, high quality illustration, no text",
        negative_prompt="scary, violent, dark horror, text, watermark, blurry, realistic photo",
        num_inference_steps=20,
        guidance_scale=7.5,
        width=512,
        height=512,
    )
    return result.images[0]


# ── XTTS-v2: 내 목소리 WAV가 있을 때만 모델을 지연 로드합니다. ──────────────
import base64
import subprocess
import time
import requests

XTTS_APP_PATH = Path("/content/test_v1_xtts_sidecar.py")
XTTS_LOG_PATH = Path("/content/test_v1_xtts_sidecar.log")
xtts_process = None

XTTS_APP_SOURCE = r"""
import base64, os, re, threading, wave
from io import BytesIO
from pathlib import Path
from tempfile import TemporaryDirectory

import torch
from fastapi import FastAPI, HTTPException
from fastapi.responses import Response
from pydantic import BaseModel
from TTS.api import TTS

app = FastAPI()
device = os.getenv('XTTS_DEVICE', 'cpu')
accepted = os.getenv('XTTS_ACCEPT_LICENSE', '').lower() == 'true'
model = None
lock = threading.Lock()

class RequestBody(BaseModel):
    text: str
    speaker_wav_b64: str

def get_model():
    global model
    if not accepted:
        raise RuntimeError('XTTS-v2 라이선스 확인값이 필요합니다.')
    if model is None:
        model = TTS(model_name='tts_models/multilingual/multi-dataset/xtts_v2', progress_bar=True).to(device)
    return model

@app.get('/health')
def health():
    return {'status':'ok', 'loaded': model is not None, 'device': device}

@app.post('/synthesize')
def synthesize(body: RequestBody):
    if not body.text.strip():
        raise HTTPException(400, '읽을 텍스트가 비어 있습니다.')
    try:
        wav = base64.b64decode(body.speaker_wav_b64, validate=True)
        if wav[:4] != b'RIFF' or wav[8:12] != b'WAVE':
            raise ValueError('PCM WAV 파일이 필요합니다.')
        with lock, TemporaryDirectory() as temp:
            speaker = Path(temp) / 'speaker.wav'
            output = Path(temp) / 'output.wav'
            speaker.write_bytes(wav)
            get_model().tts_to_file(text=body.text[:2500], speaker_wav=str(speaker), language='ko', file_path=str(output), split_sentences=True)
            return Response(output.read_bytes(), media_type='audio/wav')
    except ValueError as error:
        raise HTTPException(400, str(error))
    except Exception as error:
        raise HTTPException(503, str(error))
"""


def ensure_xtts_sidecar():
    global xtts_process
    if xtts_process is not None and xtts_process.poll() is None:
        return
    XTTS_APP_PATH.write_text(XTTS_APP_SOURCE, encoding="utf-8")
    log_file = XTTS_LOG_PATH.open("w")
    xtts_process = subprocess.Popen(
        [XTTS_PYTHON, "-m", "uvicorn", "test_v1_xtts_sidecar:app", "--host", "127.0.0.1", "--port", "8002"],
        cwd="/content",
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
    for _ in range(30):
        try:
            if requests.get("http://127.0.0.1:8002/health", timeout=2).ok:
                return
        except Exception:
            time.sleep(1)
    details = XTTS_LOG_PATH.read_text(errors="replace")[-3000:]
    raise RuntimeError("XTTS 워커 시작 실패:\n" + details)


def synthesize_with_my_voice(text, speaker_wav_path):
    speaker_path = Path(speaker_wav_path)
    if not speaker_path.is_file():
        raise FileNotFoundError(f"화자 WAV 파일이 없습니다: {speaker_path}")
    ensure_xtts_sidecar()
    response = requests.post(
        "http://127.0.0.1:8002/synthesize",
        json={"text": text, "speaker_wav_b64": base64.b64encode(speaker_path.read_bytes()).decode("ascii")},
        timeout=900,
    )
    response.raise_for_status()
    return response.content


In [ ]:
# 셀 4: 자유형 8단계 계획 생성기
# 인물의 역할 종류와 수를 코드에서 제한하지 않습니다. 모델이 이야기마다 cast를 설계합니다.
STAGE_ARCS = [
    "사건의 발생과 주인공의 목표 설정",
    "목표로 향할 수 있는 방향 또는 의미 있는 단서 확보",
    "관계·정보·도구 중 필요한 준비를 갖춤",
    "목표를 향한 행동을 본격적으로 시작하고 첫 변화를 겪음",
    "목표를 어렵게 만드는 구체적인 시험을 겪음",
    "갈등의 진실 또는 해결 실마리를 발견하고 마지막 접근을 준비함",
    "가장 큰 갈등을 지혜롭고 안전하게 해결하여 목표를 이룸",
    "사건 이후 달라진 관계와 일상을 보여 주며 완결함",
]


def normalize_cast(raw_cast):
    if isinstance(raw_cast, dict):
        raw_cast = [
            {"name": key, "description": value, "enters_at": 1}
            for key, value in raw_cast.items()
        ]
    if not isinstance(raw_cast, list):
        return []
    cast, seen = [], set()
    for item in raw_cast:
        if isinstance(item, str):
            item = {"name": item, "description": "이야기에서 중요한 역할을 한다.", "enters_at": 1}
        if not isinstance(item, dict):
            continue
        name = clean_model_text(item.get("name", ""))
        if len(name) < 1 or name in seen:
            continue
        seen.add(name)
        try:
            enters_at = int(item.get("enters_at", 1))
        except Exception:
            enters_at = 1
        cast.append({
            "name": name,
            "description": clean_model_text(item.get("description", item.get("role", ""))) or "이야기의 중요한 인물이다.",
            "desire": clean_model_text(item.get("desire", "")),
            "enters_at": max(1, min(8, enters_at)),
        })
    return cast


def normalize_stages(raw_stages):
    if not isinstance(raw_stages, list):
        return []
    stages = []
    for index, item in enumerate(raw_stages[:8], 1):
        if isinstance(item, list):
            item = {
                "title": item[0] if len(item) > 0 else "",
                "setting": item[1] if len(item) > 1 else "",
                "goal": item[2] if len(item) > 2 else "",
                "conflict": item[3] if len(item) > 3 else "",
                "progress": item[4] if len(item) > 4 else "",
            }
        if not isinstance(item, dict):
            continue
        stage = {
            "number": index,
            "arc": STAGE_ARCS[index - 1],
            "title": clean_model_text(item.get("title", "")) or f"{index}번째 장면",
            "setting": clean_model_text(item.get("setting", item.get("location", ""))),
            "goal": clean_model_text(item.get("goal", item.get("required_event", ""))),
            "question": clean_model_text(item.get("question", item.get("dramatic_question", ""))),
            "conflict": clean_model_text(item.get("conflict", item.get("obstacle", ""))),
            "progress": clean_model_text(item.get("progress", item.get("stage_result", item.get("result", "")))),
            "not_yet": clean_model_text(item.get("not_yet", item.get("must_not_happen", ""))),
            "focus_character": clean_model_text(item.get("focus_character", "")),
            "emotional_trigger": clean_model_text(item.get("emotional_trigger", "")),
            "decision_focus": clean_model_text(item.get("decision_focus", "")),
        }
        stages.append(stage)
    return stages


def normalize_plan(candidate, request):
    if not isinstance(candidate, dict):
        return None
    stages = normalize_stages(candidate.get("stages"))
    cast = normalize_cast(candidate.get("cast", candidate.get("characters", [])))
    # 내용 품질 점수로 생성 자체를 막지 않습니다. 8단계 구조와 최소 인물 목록만 확인합니다.
    if len(stages) != 8 or not cast:
        return None
    world_rules = candidate.get("world_rules", candidate.get("story_bible", []))
    if isinstance(world_rules, str):
        world_rules = [world_rules]
    if not isinstance(world_rules, list):
        world_rules = []
    threads = candidate.get("threads", candidate.get("plot_threads", []))
    if isinstance(threads, str):
        threads = [threads]
    if not isinstance(threads, list):
        threads = []
    return {
        "title": clean_model_text(candidate.get("title", "")) or "이름 없는 모험",
        "request": request,
        "lesson": clean_model_text(candidate.get("lesson", "")),
        "cast": cast,
        "world_rules": [clean_model_text(item) for item in world_rules if clean_model_text(item)],
        "threads": [clean_model_text(item) for item in threads if clean_model_text(item)],
        "stages": stages,
    }


def repair_plan_format(raw, request):
    prompt = f"""아래는 이미 작성된 동화 계획이다. 사건, 인물, 이름, 순서를 새로 만들거나 바꾸지 말고 JSON 형식만 고쳐라.

[원문]
{raw[:12000]}

반드시 아래 키를 가진 JSON 객체 하나만 출력한다.
{{
  "title":"제목",
  "lesson":"교훈",
  "cast":[{{"name":"인물 또는 중요한 존재","description":"성격과 이야기 속 기능","desire":"원하는 것","enters_at":1}}],
  "world_rules":["세계관 또는 마법 규칙"],
  "threads":["끝까지 이어질 단서 또는 관계"],
  "stages":[{{"title":"장면 제목","setting":"장소","goal":"이번 단계에서 반드시 진전될 일","question":"독자가 궁금해할 것","conflict":"장애물","progress":"장면 뒤 확정 사실","not_yet":"아직 해결하면 안 되는 것","focus_character":"이번 장면의 중심 인물","emotional_trigger":"반응을 일으키는 구체적 사건","decision_focus":"서로 다른 행동을 선택할 이유"}}]
}}
원문에 없는 focus_character, emotional_trigger, decision_focus는 빈 문자열로 둔다.
stages는 반드시 원문의 순서를 유지한 8개다. JSON 밖의 말은 쓰지 않는다."""
    repaired = call_llm(prompt, max_tokens=3200, temperature=0.05, top_p=0.8)
    return normalize_plan(extract_json_loose(repaired), request)


def make_story_plan(request, genre="판타지"):
    stage_lines = "\n".join(f"{index + 1}. {arc}" for index, arc in enumerate(STAGE_ARCS))
    prompt = f"""너는 한국어 창작동화의 기획자다. 아래 요청을 바탕으로 처음부터 끝까지 이어지는 8단계 인터랙티브 동화 계획을 만든다.

장르: {genre}
사용자 요청: {request}
문체: 어린이가 이해할 쉬운 어휘, 생생한 행동과 대화, 일관된 해요체

8단계의 서사적 기능:
{stage_lines}

중요한 자유도 규칙:
- 인물 수, 이름, 종족, 관계, 핵심 물건의 유무는 이야기 필요에 따라 자유롭게 결정한다.
- 주인공·동료·안내자·악당 같은 고정 역할 칸을 억지로 채우지 않는다. 필요한 존재만 만든다.
- 각 인물은 name, description, desire, enters_at(처음 본격적으로 등장하는 1~8 단계)를 가진다.
- 갈등 상대가 있다면 단순히 나쁜 존재로 만들지 말고 납득할 수 있는 바람이나 오해를 준다. 갈등 상대가 필요 없으면 만들지 않아도 된다.
- 세계관과 마법에는 1~4개의 구체적인 규칙을 정한다. 무엇이든 해결하는 마법은 금지한다.
- threads에는 이야기 끝까지 회수할 단서, 약속, 물건, 관계를 1~3개 넣는다.
- 1~6단계에서 최종 문제를 해결하지 않는다. 7단계에서 목표를 이루고, 8단계는 새로운 위기 없이 따뜻하게 끝낸다.
- progress는 이번 단계의 목표 결과이지 이미 일어난 사실이 아니다. 실제 선택에 따른 세부 결과는 장면에서 정한다.
- 앞 단계의 목표 결과가 다음 단계 goal로 자연스럽게 이어지게 한다. 인물은 enters_at 이전에 실제 행동을 하지 않는다.
- 특정 물건의 획득이나 특정 인물의 도움처럼 한 선택 방법만 요구하는 결과는 피한다. 같은 목표를 여러 방법으로 이룰 여지를 둔다.
- 목표·갈등·결과를 같은 문장으로 반복하지 않는다. 평범한 일상 소재에도 맞게 설계하며 여행·구출·싸움을 억지로 넣지 않는다.
- 선택에 따라 장면의 방법과 대화는 달라질 수 있지만, 8단계의 큰 사건 흐름은 유지되어야 한다.
- 각 단계의 focus_character는 그때 등장할 수 있는 cast의 인물 이름이다. emotional_trigger는 선택 방법이 바뀌어도 자연스럽게 반응을 이끌 수 있는 사건이다. 감정 이름이나 점수는 미리 배정하지 않는다.
- 1~6단계의 decision_focus에는 인물의 바람·관계·시간·자원 사이에 서로 납득할 이유가 있는 구체적인 갈림점을 쓴다.
- 7단계 decision_focus는 해결 뒤 약속을 마무리하거나 일상으로 돌아가는 방법이다. 새 단서·시험·위기를 만들지 않는다. 8단계는 빈 문자열이다.
- 갈림점에는 서로 납득할 만한 여러 방법이 있어야 한다. 한 방법만 착하고 다른 방법은 잔인하거나 어리석게 보이는 구도를 피한다.
- 단계마다 같은 고민을 반복하지 않는다. 사건이 바뀌며 인물의 반응도 자연스럽게 달라지게 한다. 분석을 위한 감정 할당량이나 정해진 선택지 유형을 만들지 않는다.
- 어린이가 이해할 수 있는 자연스러운 한국어를 사용하고 게임 퀘스트, 미션, 레벨 같은 표현은 쓰지 않는다.
- JSON 하나만 출력한다. 코드블록, 주석, 설명, 끝 쉼표는 절대 쓰지 않는다.

JSON 형식:
{{
  "title":"구체적이고 동화다운 제목",
  "lesson":"이야기 속 행동으로 드러나는 교훈",
  "cast":[{{"name":"이름","description":"성격과 이야기 속 기능","desire":"바라는 것","enters_at":1}}],
  "world_rules":["세계관 또는 마법 규칙"],
  "threads":["회수할 단서 또는 관계"],
  "stages":[
    {{"title":"장면 제목","setting":"장소","goal":"이번 단계에서 반드시 진전될 일","question":"독자가 궁금해할 것","conflict":"구체적 장애물","progress":"장면 뒤 확정 사실","not_yet":"아직 해결하면 안 되는 일","focus_character":"중심 인물 이름","emotional_trigger":"그 인물의 반응을 일으키는 사건","decision_focus":"무엇을 먼저 살필지 고민할 구체적인 갈림점"}}
  ]
}}
stages는 정확히 8개를 써야 한다."""
    raw = call_llm(prompt, max_tokens=4300, temperature=0.38, top_p=0.88)
    plan = normalize_plan(extract_json_loose(raw), request)
    if plan is None:
        print("[정보] 계획 내용은 유지한 채 JSON 형식만 한 번 복구합니다.")
        plan = repair_plan_format(raw, request)
    if plan is None:
        print("[계획 생성 원문]\n" + raw[:12000])
        raise RuntimeError("8단계 계획의 JSON 형식을 읽지 못했습니다. 위 원문을 확인하세요.")
    return plan


In [ ]:
# 셀 5: 선택지와 장면 생성. 고정 선택지 유형이나 고정 인물 역할을 사용하지 않습니다.
def stage_of(plan, number):
    return plan["stages"][number - 1]


def available_cast(plan, stage_number):
    return [item for item in plan["cast"] if item["enters_at"] <= stage_number]


def story_memory(game):
    selections = {
        item["to_stage"]: item.get("choice", {}).get("text", "")
        for item in game["state"].get("history", [])
    }
    records = []
    for scene in game["state"]["scenes"]:
        text = scene["text"]
        memory = scene.get("memory") if isinstance(scene.get("memory"), dict) else {}
        facts = source_quotes(memory.get("scene_facts"), text)
        source = "quoted_facts"
        if not facts:
            parts = sentences(text)
            facts = list(dict.fromkeys(parts[:2] + parts[-2:]))
            source = "scene_excerpt"
        records.append({
            "stage": scene["stage"],
            "selected_action": selections.get(scene["stage"], ""),
            "observed_in_scene": facts,
            "unresolved_in_scene": source_quotes(memory.get("open_threads"), text, 2),
            "source": source,
        })
    return records


def completed_progress(game):
    # 계획 결과를 달성했다고 가정하지 않고 실제 본문과 선택을 전달합니다.
    return compact_json(story_memory(game), 4200)


def plan_context(plan, current_stage):
    # 현재·다음 목표를 앞에 두고, 먼 미래는 짧은 목표만 전달합니다.
    return {
        "current_stage": stage_of(plan, current_stage),
        "next_stage": stage_of(plan, current_stage + 1) if current_stage < 8 else None,
        "title": plan["title"],
        "lesson": plan["lesson"],
        "cast_available_now": available_cast(plan, current_stage),
        "world_rules": plan["world_rules"],
        "threads": plan["threads"],
        "later_outline": [
            {"stage": item["number"], "goal": item["goal"]}
            for item in plan["stages"][current_stage + 1:]
        ],
    }


def analysis_context(game):
    stage = stage_of(game["plan"], game["state"]["stage"])
    return {
        "stage": stage["number"], "goal": stage["goal"], "conflict": stage["conflict"],
        "focus_character": stage.get("focus_character", ""),
        "decision_focus": stage.get("decision_focus", ""),
        "cast": [{"name": item["name"], "desire": item.get("desire", "")}
                 for item in available_cast(game["plan"], game["state"]["stage"])],
    }


def choice_snapshot(choice):
    return {
        "id": choice.get("id", ""), "text": choice["text"],
        "actor": choice.get("actor", ""), "decision": dict(choice.get("decision", {})),
        "origin": choice.get("origin", "legacy"),
    }


def normalize_choice_quality(data, choices):
    data = data if isinstance(data, dict) else {}
    ids = {item["id"] for item in choices}
    pairs, valid_pairs = [], True
    raw_pairs = data.get("overlapping_pairs", [])
    if not isinstance(raw_pairs, list):
        raw_pairs, valid_pairs = [], False
    for pair in raw_pairs:
        if (isinstance(pair, list) and len(pair) == 2 and
                all(isinstance(item, str) and item in ids for item in pair) and pair[0] != pair[1]):
            pairs.append(pair)
        else:
            valid_pairs = False
    seen = {}
    for choice in choices:
        action = choice["text"]
        key = re.sub(r"\W+", "", action)
        if key in seen:
            pairs.append([seen[key], choice["id"]])
        seen[key] = choice["id"]
    reason = clean_model_text(data.get("reason", ""))[:300]
    if any(item.get("origin") == "fallback" for item in choices):
        status, reason = "fallback", "진행 보완용 선택지가 포함되어 선택 경향 해석에서 제외합니다."
    elif pairs or data.get("distinct") is False:
        status = "overlap"
    elif data.get("distinct") is True and reason and valid_pairs:
        status = "model_assessed_distinct"
    else:
        status = "unknown"
    return {
        "status": status,
        "usable_for_pattern_analysis": status == "model_assessed_distinct",
        "overlapping_pairs": pairs,
        "reason": reason or "선택지 간 행동 차이를 확인할 근거가 부족합니다.",
        "source": OBSERVATION_SOURCE,
    }


def ensure_choice_observations(game):
    choices = game["state"]["choices"]
    if not choices:
        return []
    current = game["state"]["stage"]
    for index, choice in enumerate(choices, 1):
        choice["id"] = f"stage_{current}_choice_{index}"
        choice.pop("emotion", None)
    scene = game["state"]["scenes"][-1]["text"]
    context = analysis_context(game)
    offered = [choice_snapshot(choice) for choice in choices]
    key = hashlib.sha256(json.dumps(
        [OBSERVATION_SOURCE, context, scene, offered],
        ensure_ascii=False, sort_keys=True,
    ).encode("utf-8")).hexdigest()
    cached = game["state"].get("choice_assessment", {})
    if cached.get("key") == key and all(
        isinstance(choice.get("observation"), dict) and
        choice["observation"].get("source") == OBSERVATION_SOURCE
        for choice in choices
    ):
        return [choice["observation"] for choice in choices]

    prompt = f'''너는 동화 선택지 세 개의 행동 차이를 기록한다.
[현재 단계와 등장인물]
{json.dumps(context, ensure_ascii=False)}
[아이가 이미 읽은 장면]
{scene}
[제시된 선택지 전체]
{json.dumps([{"id": item["id"], "text": item["text"]} for item in offered], ensure_ascii=False)}

- 각 선택지는 id로 구별한다. 다른 선택지의 문장을 섞지 않는다.
- text에 드러난 행동만 behavior_tags에 짧게 쓴다. 숨은 동기나 아이의 성격을 추정하지 않는다.
- 감정 라벨, 감정 점수, 성향 수치, 확률을 생성하지 않는다.
- quality에서는 아이에게 보이는 text만으로 세 쌍을 비교한다. 내부 decision 메타데이터로 차이를 보충하지 않는다.
- 행동 대상·방법·우선순위의 실질적 차이가 text에 드러나는지 판단한다.
- 표현만 바뀌거나 수행할 행동이 거의 같으면 distinct=false로 하고 그 id 쌍을 overlapping_pairs에 적는다.
- 차이가 있으면 distinct=true와 구체적인 차이 reason을 쓴다. 번호별 성격이나 고정 선택 유형을 가정하지 않는다.
JSON만 출력한다.
{{"choices":[{{"id":"선택지 id","behavior_tags":["행동 요약"]}}],"quality":{{"distinct":true,"overlapping_pairs":[],"reason":"세 행동 사이의 구체적 차이"}}}}'''
    data = request_observation_json(prompt, 800)
    results = data.get("choices", [])
    results = results if isinstance(results, list) else []
    by_id, duplicates = {}, set()
    for item in results:
        if not isinstance(item, dict) or not isinstance(item.get("id"), str):
            continue
        choice_id = item["id"]
        if choice_id in by_id:
            duplicates.add(choice_id)
        by_id[choice_id] = item
    quality = normalize_choice_quality(data.get("quality"), offered)
    for choice in choices:
        raw = by_id.get(choice["id"], {}) if choice["id"] not in duplicates else {}
        choice["observation"] = behavior_observation(raw)
    game["state"]["choice_assessment"] = {"key": key, **quality}
    return [choice["observation"] for choice in choices]


def recent_choice_context(game):
    return [
        {"stage": item["from_stage"], "offered": [
            {"text": option["text"], "priority": option.get("decision", {}).get("priority", "")}
            for option in item.get("offered_choices", [])
        ]}
        for item in game["state"]["history"][-2:]
    ]


def normalize_choices(candidate, raw_text):
    items = candidate.get("choices", []) if isinstance(candidate, dict) else candidate
    if not isinstance(items, list):
        items = []
    choices, seen = [], set()
    for item in items:
        if isinstance(item, str):
            item = {"text": item}
        if not isinstance(item, dict):
            continue
        text = clean_model_text(item.get("short_text") or item.get("text", ""))
        key = re.sub(r"\W+", "", text)
        if len(text) < 4 or len(text) > 40 or key in seen:
            continue
        seen.add(key)
        decision = item.get("decision", {})
        decision = decision if isinstance(decision, dict) else {}
        choices.append({
            "text": text,
            "actor": clean_model_text(item.get("actor", "")),
            "decision": {name: clean_model_text(decision.get(name, "")) for name in ("action", "priority", "tradeoff")},
            "origin": "model",
            "opening": clean_model_text(item.get("opening", "")),
            "consequence": clean_model_text(item.get("consequence", item.get("bridge", ""))),
        })
        if len(choices) == 3:
            return choices

    # 닫히지 않은 JSON에서도 완전한 text 문자열은 그대로 복구합니다.
    for match in re.finditer(r'"(?:text|short_text)"\s*:\s*("(?:\\.|[^"\\])*")', raw_text):
        try:
            text = clean_model_text(json.loads(match.group(1)))
        except (ValueError, TypeError):
            continue
        key = re.sub(r"\W+", "", text)
        if 4 <= len(text) <= 40 and key not in seen:
            seen.add(key)
            choices.append({"text": text, "opening": "", "consequence": "",
                            "decision": {}, "actor": "", "origin": "text_recovered"})
        if len(choices) == 3:
            return choices

    # 번호 목록도 모델 재호출 없이 복구합니다.
    for line in clean_model_text(raw_text).splitlines():
        text = re.sub(r"^\s*(?:[-*]|\d+[.)])\s*", "", line).strip()
        key = re.sub(r"\W+", "", text)
        if (5 <= len(text) <= 40 and key not in seen and
                not re.search(r'[{}\[\]`:]|"(?:text|opening|consequence|decision|actor)"', text)):
            seen.add(key)
            choices.append({"text": text, "opening": "", "consequence": "", "decision": {}, "actor": "", "origin": "text_recovered"})
        if len(choices) == 3:
            break
    return choices


def repair_choice_format(raw, context=None):
    return normalize_choices(extract_json_loose(raw), raw)


def complete_choice_list(choices):
    # 형식을 전혀 읽지 못했을 때만 읽기를 계속할 수 있는 중립 항목을 보완합니다.
    continuation = [
        "다음 장면으로 이야기를 이어간다.",
        "주인공의 다음 행동을 따라가 본다.",
        "이어지는 장면에서 무슨 일이 생기는지 읽는다.",
    ]
    result = list(choices)
    known = {re.sub(r"\W+", "", item["text"]) for item in result}
    for text in continuation:
        if len(result) >= 3:
            break
        key = re.sub(r"\W+", "", text)
        if key not in known:
            known.add(key)
            result.append({"text": text, "actor": "", "decision": {},
                           "opening": "", "consequence": "", "origin": "fallback"})
    return result[:3]


def model_choices(game):
    current = game["state"]["stage"]
    if current >= 8:
        return []
    context = plan_context(game["plan"], current)
    last_scene = game["state"]["scenes"][-1]["text"]
    close_rule = (
        "가장 큰 갈등은 이미 풀렸다. 약속을 지키거나 관계를 마무리하거나 일상으로 돌아가는 구체적인 방법을 고른다. 새 위기와 새 단서를 만들지 않는다."
        if current == 7 else
        "이번 장면의 갈림점에서 다음 단계 목표로 다가가는 서로 다른 방법을 고른다."
    )
    prompt = f"""너는 한국어 인터랙티브 동화의 선택지 작가다.
장르: {game.get('settings', {}).get('genre', '판타지')}

[현재 목표와 다음 목표, 등장한 인물]
{compact_json(context, 4200)}

[실제 본문과 선택에서 남은 기록]
{completed_progress(game)}

[독자가 방금 읽은 장면]
{last_scene}

[최근 두 단계에서 제시했던 행동]
{compact_json(recent_choice_context(game), 1100)}

선택지 규칙:
- 정확히 세 개를 쓴다. 등장한 인물·장소·물건과 지금 해결할 문제에 맞는 실행 가능한 행동이다.
- 세 선택의 행동 대상, 방법 또는 우선순위가 화면에 보이는 text 자체에서 구분되어야 한다. 숨겨진 분석 정보만 다르게 쓰지 않는다.
- 같은 행동을 동사만 바꿔 세 번 쓰지 않는다. 어떤 방법이 적절한지는 이 장면을 보고 결정하며 번호별 유형은 고정하지 않는다.
- 각 방법을 고를 만한 이유가 있어야 한다. 정답처럼 보이는 한 선택과 어리석은 두 선택을 만들지 않는다.
- 작은 부담이나 미뤄야 할 일이 실제로 있는 경우에만 tradeoff를 적는다. 모든 선택에 위험이나 손해를 억지로 만들지 않는다.
- decision은 text에서 읽을 수 있는 행동·우선순위만 요약한다. text에 없는 동기·감정·성격을 추정하지 않는다.
- text는 12~40자의 쉬운 현재형 한 문장이다. 이름을 포함하느라 길어지면 맥락상 분명한 주어는 생략해도 된다.
- actor는 현재 등장한 인물의 이름이다. 낯선 인물·물건·해결 능력을 추가하지 않는다.
- "용감하게/친절하게/겁먹고" 같은 평가를 붙이지 않는다. 구체적인 행동으로 표현한다.
- 다음 장면의 본문과 결과를 미리 쓰지 않는다. 이 선택으로 무엇을 먼저 할지만 결정한다.
- 이미 선택으로 확정되거나 본문에서 달성된 사실을 취소하지 않는다.
- {close_rule}
- 이모지·번호·주석·코드블록 없이 JSON만 출력한다.

{{"choices":[
{{"text":"첫 구체적인 행동","actor":"인물 이름","decision":{{"action":"text의 실제 행동","priority":"text에 드러난 우선순위","tradeoff":""}}}},
{{"text":"다른 구체적인 행동","actor":"인물 이름","decision":{{"action":"text의 실제 행동","priority":"text에 드러난 우선순위","tradeoff":""}}}},
{{"text":"또 다른 구체적인 행동","actor":"인물 이름","decision":{{"action":"text의 실제 행동","priority":"text에 드러난 우선순위","tradeoff":""}}}}
]}}"""
    prompt = fit_generation_prompt(prompt, [
        ("현재 목표와 다음 목표, 등장한 인물", context, 4200, 1800),
        ("실제 본문과 선택에서 남은 기록", story_memory(game), 4200, 1000),
        ("독자가 방금 읽은 장면", last_scene, len(last_scene), 600),
        ("최근 두 단계에서 제시했던 행동", recent_choice_context(game), 1100, 300),
    ], 1100)
    raw = call_llm(prompt, max_tokens=1100, temperature=0.62, top_p=0.92)
    choices = repair_choice_format(raw)
    if len(choices) < 3:
        print("[정보] 복구 가능한 선택지는 유지하고 읽기용 항목을 보완합니다.")
    choices = complete_choice_list(choices)
    for index, choice in enumerate(choices, 1):
        choice["id"] = f"stage_{current}_choice_{index}"
    return choices


def write_scene(game, stage_number, selected=None):
    plan = game["plan"]
    context = plan_context(plan, stage_number)
    current = stage_of(plan, stage_number)
    previous = game["state"]["scenes"][-1]["text"] if game["state"]["scenes"] else "이야기의 시작이다."
    bound_action = "첫 장면이므로 이전 선택은 없다."
    if selected:
        bound_action = selected["text"]
    stage_end_rule = (
        "이번 단계에서 가장 큰 갈등을 실제 행동과 대화로 해결하고 목표가 이루어졌음을 독자가 분명히 알게 한다."
        if stage_number == 7 else
        "새 위기를 만들지 말고 해결 뒤 달라진 관계와 일상을 보여 주며 마지막 문장에서 이야기를 완전히 끝낸다."
        if stage_number == 8 else
        "이번 단계의 진전까지만 보여 주고 이후 단계의 핵심 해결을 앞당기지 않는다."
    )
    prompt = f"""너는 쉬운 어휘와 생생한 장면으로 자연스러운 한국어 창작동화를 쓰는 작가다.
장르: {game.get('settings', {}).get('genre', '판타지')}

[전체 계획]
{compact_json(context)}

[실제 본문과 선택에서 남은 기록]
{completed_progress(game)}

[직전 장면]
{previous}

[선택으로 확정된 이번 장면의 첫 행동]
{bound_action}

[이번 장면]
- 단계: {stage_number}/8
- 제목: {current['title']}
- 장소: {current['setting']}
- 이번 단계 목표: {current['goal']}
- 독자의 질문: {current['question']}
- 장애물 또는 갈등: {current['conflict']}
- 이번 장면에서 이룰 목표 결과: {current['progress']}
- 아직 일어나면 안 되는 일: {current['not_yet']}
- 주로 반응을 따라갈 인물: {current.get('focus_character', '')}
- 반응을 이끌 계획상의 사건: {current.get('emotional_trigger', '')}
- 장면 끝에서 고민할 갈림점: {current.get('decision_focus', '')}

작성 규칙:
- 이전 선택이 있다면 첫 문단에서 실제로 실행한다. 선택 문장을 그대로 복사하지 말고 해요체의 행동과 대화로 자연스럽게 보여 준다.
- 계획의 목표 결과는 아직 일어난 사실이 아니다. 실제 기록과 선택 행동에서 출발해 이번 장면 안에서 목표를 이룬다.
- 선택이 읽기를 계속하는 중립 항목이면 현재 단계에 맞는 자연스러운 첫 행동을 만든다. "다음 장면을 읽었어요" 같은 앱 조작을 동화 사건으로 쓰지 않는다.
- 기록 속 물건의 획득·분실, 인물이 알고 있는 정보, 이미 지킨 약속을 이유 없이 바꾸지 않는다. 오래된 기록이 뒤 장면과 달라지면 더 최근의 실제 변화를 따른다.
- 선택한 방법 때문에 얻는 정보·도움·물건의 상태나 작은 결과가 달라지고, 그 결과가 이번 단계의 진전으로 이어지게 쓴다.
- 계획의 중심 사건과 최종 목표는 지키되, 특정 행동 방식과 예정된 감정 반응은 선택과 실제 사건에 맞게 조정한다.
- 계획의 cast에 정해진 인물·중요 존재만 실제 이름을 갖고 등장하거나 행동하게 한다. 이름 없는 군중·동물·마을 사람은 분위기 묘사로만 짧게 쓸 수 있으며, 새 이름·새 핵심 인물·새 핵심 물건을 만들지 않는다.
- 모델이 계획한 인물의 성격, 바람, 등장 시점을 끝까지 지킨다.
- 세계 규칙과 이미 나온 단서를 잊지 말고, 계획표를 독자에게 설명하지 않는다.
- 행동, 감각 묘사, 짧은 대화로 장면을 보여 준다. 같은 격려·감탄·문장을 반복하지 않는다.
- 중요한 사건에 대해 누가, 왜, 어떻게 반응했는지 인물 이름·몸짓·대사로 알아볼 수 있게 쓴다. 여러 인물의 서로 다른 반응을 한 감정으로 합치지 않는다.
- 선택의 결과가 물건·정보·관계에 실제로 작은 변화를 남기게 한다. 다음 단계로 넘어간다는 설명만으로 결과를 대신하지 않는다.
- 인물의 바람과 사건에서 자연스럽게 생긴 반응을 보여 준다. 선택 감정 점수에 맞춰 마음을 강제하거나, "이 선택은 용기/친절"이라고 평가하지 않는다.
- 감정 단어를 문장마다 붙이거나 독자의 감정을 대신 단정하지 않는다. 필요할 때만 쉬운 말로 인물의 마음을 짧게 드러낸다.
- 1~6단계 끝에는 등장한 대상·정보를 바탕으로 여러 행동이 가능한 구체적인 상황을 남긴다.
- 7단계는 목표를 이룬 뒤 마무리 방법을 고를 여지를 남기며 새 위기나 해결을 되돌리는 의문을 만들지 않는다.
- 8단계는 갈림점과 질문 예고 없이 완결한다. 본문에 선택지 목록을 쓰지 않는다.
- 했어요, 그랬어요, 했답니다처럼 부드러운 해요체를 일관되게 사용한다.
- 잔혹한 폭력, 죽음, 과도한 공포, 게임 용어, 이모지, 제목, 단계 번호, 해시태그, 코드, 다음 장 예고를 쓰지 않는다.
- 3~4문단, 12~18문장으로 충분히 길고 매끄럽게 쓴다.
- {stage_end_rule}

동화 본문만 출력하라."""
    prompt = fit_generation_prompt(prompt, [
        ("전체 계획", context, 5200, 1800),
        ("실제 본문과 선택에서 남은 기록", story_memory(game), 4200, 1000),
        ("직전 장면", previous, len(previous), 600),
    ], 1250)
    raw = call_llm(prompt, max_tokens=1250, temperature=0.66, top_p=0.91)
    text = clean_story(raw, headings=[plan["title"], current["title"]])
    return text


In [ ]:
# 셀 6: 선택 당시의 전체 후보와 장면 참조를 보존하는 상태 머신
import copy
import random
import secrets


def new_game(plan, genre="판타지"):
    return {
        "record_id": "story_" + secrets.token_urlsafe(12),
        "plan": copy.deepcopy(plan),
        "settings": {"genre": genre, "run_mode": "interactive", "emotion_scoring_enabled": False},
        "state": {"stage": 1, "ended": False, "scenes": [], "choices": [],
                  "history": [], "choice_assessment": {}},
    }


def start_game(plan, genre="판타지"):
    game = new_game(plan, genre)
    text = write_scene(game, 1)
    game["state"]["scenes"].append({
        "stage": 1, "title": stage_of(plan, 1)["title"], "text": text,
        "memory": extract_scene_memory(text, game),
    })
    game["state"]["choices"] = model_choices(game)
    ensure_choice_observations(game)
    return game


def all_story_text(game):
    return "\n\n".join(scene["text"] for scene in game["state"]["scenes"])


def choose_and_advance(game, choice_index):
    if choice_index not in (0, 1, 2):
        raise ValueError("선택지 번호는 0, 1, 2 중 하나여야 합니다.")
    if game["state"]["ended"]:
        raise ValueError("이미 8단계 결말까지 완료된 동화입니다.")
    discard_emotion_scores(game)
    ensure_choice_observations(game)
    updated = copy.deepcopy(game)
    current = updated["state"]["stage"]
    offered = [
        {**choice_snapshot(item), "observation": copy.deepcopy(item.get("observation", {}))}
        for item in updated["state"]["choices"]
    ]
    selected = copy.deepcopy(updated["state"]["choices"][choice_index])
    assessment = {
        key: value for key, value in updated["state"].get("choice_assessment", {}).items()
        if key != "key"
    }
    next_stage = current + 1
    updated["state"]["stage"] = next_stage
    text = write_scene(updated, next_stage, selected)
    updated["state"]["scenes"].append({
        "stage": next_stage, "title": stage_of(updated["plan"], next_stage)["title"],
        "text": text, "memory": extract_scene_memory(text, updated),
    })
    updated["state"]["history"].append({
        "from_stage": current, "to_stage": next_stage,
        "context_scene_stage": current,
        "stage_goal": stage_of(updated["plan"], current)["goal"],
        "offered_choices": offered, "selected_choice_id": selected["id"],
        "choice": selected, "decision_quality": assessment,
    })
    updated["state"]["choice_assessment"] = {}
    if next_stage == 8:
        updated["state"]["ended"] = True
        updated["state"]["choices"] = []
    else:
        updated["state"]["choices"] = model_choices(updated)
        ensure_choice_observations(updated)
    return updated


def decision_training_examples(game):
    # 모델 라벨은 초안입니다. 미래의 이어쓰기 본문은 분류 입력에 넣지 않습니다.
    discard_emotion_scores(game)
    scenes = {item["stage"]: item["text"] for item in game["state"]["scenes"]}
    group_id = game.get("record_id") or hashlib.sha256(
        json.dumps(game["plan"], ensure_ascii=False, sort_keys=True).encode("utf-8")
    ).hexdigest()[:24]
    examples = []
    for item in game["state"]["history"]:
        step = item["from_stage"]
        choice = item["choice"]
        offered = item.get("offered_choices", [])
        examples.append({
            "schema_version": 2,
            "story_id": group_id,
            "stage": step,
            "run_mode": game.get("settings", {}).get("run_mode", "unknown"),
            "input": {
                "stage_goal": item.get("stage_goal") or stage_of(game["plan"], step)["goal"],
                "scene": scenes.get(item.get("context_scene_stage", step), ""),
                "cast": [person["name"] for person in available_cast(game["plan"], step)],
                "choices": [{"id": option["id"], "text": option["text"]} for option in offered],
                "selected_choice_id": item.get("selected_choice_id"),
                "selected_choice_text": choice["text"],
            },
            "decision_quality": copy.deepcopy(item.get("decision_quality", {})),
            "model_annotation": {
                "source": OBSERVATION_SOURCE,
                "choice": copy.deepcopy(choice.get("observation", {})),
                "options": copy.deepcopy(offered),
            },
            "label_status": "needs_human_review",
            "human_labels": None,
        })
    return examples



In [ ]:
# 셀 7: Flutter 연동 FastAPI 서버. 셀 1~6 실행 뒤 이 셀을 실행하면 됩니다.
# 같은 메모리의 Qwen 계획 엔진을 사용하므로 별도의 모델 로딩이 필요하지 않습니다.

import base64
import os
from concurrent.futures import ThreadPoolExecutor
import re
import secrets
import subprocess
import threading
import time
from io import BytesIO
from tempfile import NamedTemporaryFile, TemporaryDirectory

import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, Response


ACTIVE_GAMES = {}
START_JOBS = {}
START_JOBS_LOCK = threading.Lock()
START_JOB_EXECUTOR = ThreadPoolExecutor(max_workers=1, thread_name_prefix='story-start')
STT_MODEL = None
STT_MODEL_LOCK = threading.Lock()


def ensure_stt_model():
    global STT_MODEL
    if STT_MODEL is not None:
        return STT_MODEL
    with STT_MODEL_LOCK:
        if STT_MODEL is None:
            from faster_whisper import WhisperModel
            STT_MODEL = WhisperModel(
                STT_MODEL_NAME,
                device=STT_DEVICE,
                compute_type=STT_COMPUTE_TYPE,
                download_root=STT_CACHE_ROOT,
            )
    return STT_MODEL


def encode_runtime_state(game):
    discard_emotion_scores(game)
    raw = json.dumps(
        {"version": 1, "game": game},
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")
    return base64.urlsafe_b64encode(raw).decode("ascii")


def decode_runtime_state(token):
    if not isinstance(token, str) or not token.strip():
        return None
    try:
        raw = base64.urlsafe_b64decode(token.encode("ascii"))
        if len(raw) > 250_000:
            return None
        payload = json.loads(raw.decode("utf-8"))
        game = payload.get("game") if isinstance(payload, dict) else None
        if not isinstance(game, dict):
            return None
        if not isinstance(game.get("plan"), dict) or not isinstance(game.get("state"), dict):
            return None
        if not isinstance(game["state"].get("scenes"), list):
            return None
        return discard_emotion_scores(game)
    except Exception:
        return None


api_app = FastAPI(title="Fairytale test_v1 API")
api_app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)


def api_error(status, message):
    raise HTTPException(status_code=status, detail=message)


def story_characters_payload(game):
    emojis = ["🌟", "🦊", "🦉", "🐉", "🌿", "✨", "🗺️"]
    profiles = []
    for index, item in enumerate(game["plan"]["cast"][:5]):
        profiles.append({
            "name": item["name"],
            "role": "이야기 속 중요한 인물",
            "personality": item["description"],
            "greeting": f"안녕, 나는 {item['name']}이야. 우리 이야기에서 궁금한 것이 있니?",
            "avatar_emoji": emojis[index % len(emojis)],
        })
    return profiles


def game_for(story_id, runtime_state=None):
    game = ACTIVE_GAMES.get(story_id)
    if game is not None:
        return discard_emotion_scores(game)
    game = decode_runtime_state(runtime_state)
    if game is None:
        api_error(404, "이어 읽기 정보가 없어요. 새 동화를 먼저 시작해 주세요.")
    ACTIVE_GAMES[story_id] = game
    return game


def response_for_start(story_id, game):
    first = game["state"]["scenes"][0]
    discard_emotion_scores(game)
    ensure_choice_observations(game)
    return {
        "story_id": story_id,
        "story_text": first["text"],
        "choices": [item["text"] for item in game["state"]["choices"]],
        "choice_emotions": [],
        "story_emotion": None,
        "emotion_scoring_enabled": False,
        "choice_metadata": [choice_snapshot(item) for item in game["state"]["choices"]],
        "choice_assessment": game["state"].get("choice_assessment", {}),
        "vocab": [],
        "chapter": 1,
        "image_b64": None,
        "characters": {item["name"]: item["description"] for item in game["plan"]["cast"]},
        "runtime_state": encode_runtime_state(game),
        "completed": False,
    }


@api_app.get("/")
def api_root():
    return {"status": "ok", "service": "fairytale test_v1"}


@api_app.get("/health")
def api_health():
    return {
        "status": "ok",
        "engine": "Qwen3.8-27B 4-bit + INT4/SNQK position-compressed kernel (LoRA off)",
        "emotion_scoring_enabled": False,
        "active_stories": len(ACTIVE_GAMES),
        "tts": "XTTS-v2 voice cloning",
        "stt": f"faster-whisper {STT_MODEL_NAME} ({'ready' if STT_MODEL is not None else 'lazy'})",
    }


def build_start_story_job(job_id, body):
    try:
        prompt = clean_model_text(body.get("prompt", "")) or "따뜻한 모험 동화"
        genre = clean_model_text(body.get("genre", "")) or "판타지"
        plan = make_story_plan(prompt, genre=genre)
        game = start_game(plan, genre)
        story_id = "testv1_" + secrets.token_urlsafe(12)
        ACTIVE_GAMES[story_id] = game
        result = response_for_start(story_id, game)
        with START_JOBS_LOCK:
            START_JOBS[job_id] = {
                **START_JOBS[job_id], "status": "completed", "result": result,
            }
    except Exception as error:
        print(f"동화 생성 작업 실패 ({job_id}): {error}", flush=True)
        with START_JOBS_LOCK:
            START_JOBS[job_id] = {
                **START_JOBS[job_id], "status": "failed", "error": str(error),
            }


@api_app.post("/story/start")
async def api_start(request: Request):
    body = await request.json()
    job_id = secrets.token_urlsafe(12)
    with START_JOBS_LOCK:
        expired = [
            key for key, value in START_JOBS.items()
            if value.get("status") != "processing"
            and time.time() - value.get("created_at", time.time()) > 3600
        ]
        for key in expired:
            START_JOBS.pop(key, None)
        START_JOBS[job_id] = {"status": "processing", "created_at": time.time()}
    START_JOB_EXECUTOR.submit(build_start_story_job, job_id, body)
    return JSONResponse(status_code=202, content={"job_id": job_id, "status": "processing"})


@api_app.get("/story/start/jobs/{job_id}")
async def api_start_job_status(job_id: str):
    with START_JOBS_LOCK:
        job = START_JOBS.get(job_id)
        if job is None:
            api_error(404, "동화 생성 요청을 찾을 수 없습니다. 다시 시도해 주세요.")
        return dict(job)


@api_app.post("/story/continue")
async def api_continue(request: Request):
    body = await request.json()
    story_id = str(body.get("story_id", ""))
    game = game_for(story_id, body.get("runtime_state"))
    if game["state"]["ended"]:
        api_error(400, "이미 완결된 동화입니다.")
    requested_choice = clean_model_text(body.get("choice", ""))
    current_choices = game["state"]["choices"]
    choice_index = next((index for index, item in enumerate(current_choices) if item["text"] == requested_choice), None)
    if choice_index is None:
        api_error(400, "현재 장면의 선택지를 골라 주세요.")
    updated = choose_and_advance(game, choice_index)
    ACTIVE_GAMES[story_id] = updated
    scene = updated["state"]["scenes"][-1]
    return {
        "story_id": story_id,
        "new_text": scene["text"],
        "choices": [item["text"] for item in updated["state"]["choices"]],
        "choice_emotions": [],
        "selected_choice_emotion": None,
        "story_emotion": None,
        "emotion_scoring_enabled": False,
        "choice_metadata": [choice_snapshot(item) for item in updated["state"]["choices"]],
        "choice_assessment": updated["state"].get("choice_assessment", {}),
        "decision_record": updated["state"]["history"][-1],
        "vocab": [],
        "chapter": updated["state"]["stage"],
        "image_b64": None,
        "runtime_state": encode_runtime_state(updated),
        "completed": updated["state"]["ended"],
    }


def _psych_story_context(game):
    if game is None:
        return {"records": [], "run_mode": "unknown", "context_available": False}
    records = []
    for row in decision_training_examples(game):
        inputs = row["input"]
        scene = inputs["scene"]
        if len(scene) > 300:
            scene = scene[:140] + " … " + scene[-140:]
        records.append({
            "step": row["stage"], "goal": inputs["stage_goal"][:100],
            "scene_excerpt": scene,
            "offered_choices": [
                {"id": option["id"], "text": option["text"],
                 "priority": option.get("decision", {}).get("priority", "")[:50],
                 "behavior_tags": option.get("observation", {}).get("behavior_tags", [])}
                for option in row["model_annotation"]["options"]
            ],
            "selected_choice_id": inputs["selected_choice_id"],
            "choice": inputs["selected_choice_text"],
            "decision_quality": row["decision_quality"],
        })
    return {"records": records, "run_mode": game.get("settings", {}).get("run_mode", "unknown"),
            "context_available": True}


def psych_coverage(context):
    context = context if isinstance(context, dict) else {}
    records = context.get("records", [])
    return {
        "total_decisions": len(records),
        "comparable_decisions": sum(
            bool(item.get("offered_choices")) and
            item.get("decision_quality", {}).get("usable_for_pattern_analysis") is True
            for item in records
        ),
        "run_mode": context.get("run_mode", "unknown"),
    }


def api_psych_prompt(title, choices, emotions, story_context):
    context = story_context if isinstance(story_context, dict) else {}
    evidence = context.get("records", [])
    if not evidence:
        evidence = [
            {"step": index + 1, "choice": clean_model_text(choice),
             "offered_choices": [],
             "decision_quality": {"usable_for_pattern_analysis": False, "status": "missing_options"}}
            for index, choice in enumerate(choices)
        ]
    coverage = psych_coverage({**context, "records": evidence})
    return f'''너는 아동 발달과 놀이 관찰 원칙에 따라 동화 속 선택을 설명하는 해설가다.
실제 아이의 성격·지능·정신건강을 판단하거나 진단하는 검사가 아니다.

[동화 제목]
{title}
[분석 가능한 기록 수와 실행 방식]
{json.dumps(coverage, ensure_ascii=False)}
[장면, 제시된 모든 대안, 실제 선택, 인용 근거]
{json.dumps(evidence, ensure_ascii=False)}

작성 원칙:
- 실제 선택 문장을 2개 이상 짧게 인용한다. 기록이 한 개면 한 개만 사용한다.
- 선택을 설명할 때 그때 제시된 다른 대안과 비교해 무엇을 우선했는지 말한다. choice 텍스트가 기준이며 내부 priority와 behavior_tags는 모델의 잠정 해석이다.
- 감정 점수를 입력으로 사용하지 않는다. 본문에 직접 나타난 인물의 반응은 문장으로 설명하되 아이의 감정으로 바꾸지 않는다.
- 행동과 감정은 별개다. 혼자 움직인 사실만으로 용감함·불안함을 단정하지 않는다.
- usable_for_pattern_analysis=false인 단계는 선택한 사실만 기술한다. 비슷한 대안, 임시 선택지, 대안이 없는 기록을 반복 선호의 근거로 세지 않는다.
- 누적 경향은 서로 구분되는 대안이 있었던 여러 단계에서만 조심스럽게 해석한다. 제시 횟수와 장면 차이를 고려하며 단순 감정 합계로 성격을 판단하지 않는다.
- comparable_decisions가 2 미만이면 반복 경향을 주장하지 않는다.
- run_mode=random_demo는 컴퓨터의 무작위 테스트다. 아이가 실제 선택했다거나 선호한다고 해석하지 않는다.
- 미래의 생성된 결과를 아이가 선택한 이유였다고 거꾸로 해석하지 않는다. 고른 이유는 열린 질문으로 확인한다.
- 감정 점수, 성향 점수, 확률, 퍼센트, 등급을 생성하지 않는다. traits는 빈 객체로 둔다.
- '따뜻한 여정', '세계와 소통' 같은 일반 칭찬 대신 관찰 사실과 가능한 의미를 구분한다.
- description은 [동화 속 관찰], [가능한 의미], [함께 이야기해 보기] 소제목의 7~9문장이다. 근거가 부족하면 그 한계를 구체적으로 쓴다.
- choice_insights에는 각 선택을 순서대로 '관찰: ... / 가능한 의미: ...'로 쓴다.
- caregiver_prompts에는 정답을 유도하지 않는 열린 질문 3개를 넣는다.
- analysis_caution에는 동화 속 인물의 정서 및 행동 해석이지 아이의 실제 감정·성격 진단이 아니라는 점을 쓴다.
- JSON 외의 말이나 코드블록은 쓰지 않는다.
{{"type":"동화 속 선택 관찰","description":"해설","traits":{{}},"dominant_emotions":[],"choice_insights":[],"caregiver_prompts":[],"analysis_caution":"심리검사가 아닌 대화용 해설입니다."}}'''


def normalize_api_psych(raw, choices, emotions, context=None):
    data = extract_json_loose(raw)
    if not isinstance(data, dict):
        data = {}
    coverage = psych_coverage(context)
    insights = data.get("choice_insights") if isinstance(data.get("choice_insights"), list) else []
    insights = [clean_model_text(item) for item in insights if clean_model_text(item)]
    if not insights:
        insights = [
            f"관찰: '{choice}'을 골랐어요. / 가능한 의미: 이 장면에서 어떤 방법을 먼저 살피고 싶었는지 함께 물어볼 수 있어요."
            for choice in choices
        ]
    description = clean_model_text(data.get("description", ""))
    if len(description) < 80:
        first = clean_model_text(choices[0]) if choices else "다음 길을 고른 장면"
        second = clean_model_text(choices[1]) if len(choices) > 1 else first
        description = (
            f"[동화 속 관찰] 주인공은 '{first}'과 '{second}'을 선택하며 장면을 이어 갔어요. "
            "이 기록에서는 선택한 행동을 확인할 수 있으며 마음은 직접 물어보아야 해요.\n\n"
            "[가능한 의미] 이 기록은 이번 동화에서 어떤 해결 방법에 마음이 갔는지 대화로 살펴볼 단서가 될 수 있어요. "
            "한 편의 동화만으로 현실의 성격을 판단할 수는 없어요.\n\n"
            "[함께 이야기해 보기] '그 선택을 고른다면 다음에는 어떤 일이 일어날 것 같았어?'라고 편하게 물어보세요."
        )
    prompts = data.get("caregiver_prompts") if isinstance(data.get("caregiver_prompts"), list) else []
    prompts = [clean_model_text(item) for item in prompts if clean_model_text(item)][:3]
    if len(prompts) < 3:
        prompts = [
            "이야기에서 가장 오래 생각한 선택은 무엇이었어?",
            "그 선택을 고르면 다음에 어떤 일이 생길 것 같았어?",
            "다시 읽는다면 다른 방법도 떠오르니?",
        ]
    return {
        "type": clean_model_text(data.get("type", "")) or "동화 속 선택 관찰",
        "description": description,
        "traits": {},
        "dominant_emotions": [],
        "emotion_scoring_enabled": False,
        "analysis_coverage": coverage,
        "choice_insights": insights[:max(1, len(choices))],
        "caregiver_prompts": prompts,
        "analysis_caution": clean_model_text(data.get("analysis_caution", "")) or "동화 속 선택을 바탕으로 한 대화용 해설이며 심리검사가 아닙니다.",
    }


@api_app.post("/story/psych")
async def api_psych(request: Request):
    body = await request.json()
    story_id = str(body.get("story_id", ""))
    game = ACTIVE_GAMES.get(story_id)
    if game is None and body.get("runtime_state"):
        game = decode_runtime_state(body["runtime_state"])
    choices = body.get("choices_made") if isinstance(body.get("choices_made"), list) else []
    emotions = []  # 기존 클라이언트가 보낸 감정 점수도 입력으로 사용하지 않습니다.
    if game is not None:
        selected = [item["choice"] for item in game["state"].get("history", [])]
        choices = [item["text"] for item in selected]
    context = _psych_story_context(game)
    raw = call_llm(
        api_psych_prompt(
            clean_model_text(body.get("story_title", "")) or "완성된 동화",
            choices,
            emotions,
            context,
        ),
        1250,
        0.25,
        0.85,
    )
    return normalize_api_psych(raw, choices, emotions, context=context)


@api_app.post("/story/characters")
async def api_characters(request: Request):
    body = await request.json()
    game = game_for(str(body.get("story_id", "")))
    return {"characters": story_characters_payload(game)}


def character_chat_context(body):
    # Live games use the full plan; saved and mock stories use Flutter context.
    story_id = str(body.get("story_id", ""))
    game = ACTIVE_GAMES.get(story_id)
    character = body.get("character") if isinstance(body.get("character"), dict) else {}
    character_name = clean_model_text(character.get("name", ""))
    if not character_name:
        api_error(400, "대화할 캐릭터 이름이 필요합니다.")

    if game is not None:
        profile = next((item for item in game["plan"]["cast"] if item["name"] == character_name), None)
        if profile is None:
            api_error(400, "이 동화 계획에 없는 캐릭터입니다.")
        return profile, game["plan"], all_story_text(game)[-11000:], "현재 서버의 동화 계획"

    # The Flutter client includes these values so a server restart must not
    # turn an already-read story into a generic local-only chat.
    story_text = clean_model_text(body.get("story_text", ""))[-12000:]
    if not story_text:
        api_error(400, "서버에 동화 기록이 없고 전달된 동화 본문도 비어 있습니다.")
    profile = {
        "name": character_name,
        "role": clean_model_text(character.get("role", "")) or "이야기 속 친구",
        "description": clean_model_text(character.get("personality", "")) or "동화의 사건을 함께 겪은 다정한 친구",
        "greeting": clean_model_text(character.get("greeting", "")),
    }
    recovered_plan = {
        "title": clean_model_text(body.get("story_title", "")) or "저장된 동화",
        "cast": [profile],
        "note": "서버 재시작 뒤 Flutter가 전달한 완성 동화 본문으로 복원한 대화 맥락입니다.",
    }
    return profile, recovered_plan, story_text, "Flutter가 전달한 저장 동화 본문"


@api_app.post("/story/character-chat")
async def api_character_chat(request: Request):
    body = await request.json()
    profile, plan_context, story_text, context_source = character_chat_context(body)
    character_name = profile["name"]
    messages = body.get("messages") if isinstance(body.get("messages"), list) else []
    user_name = clean_model_text(body.get("user_name", "")) or "동화 친구"
    user_message = clean_model_text(body.get("user_message", ""))
    if not user_message:
        api_error(400, "보낼 메시지를 입력해 주세요.")
    history_text = "\n".join(
        f"{'아이' if item.get('role') == 'user' else character_name}: {clean_model_text(item.get('content', ''))[:400]}"
        for item in messages[-12:] if isinstance(item, dict)
    ) or "아직 대화 없음"
    prompt = f'''너는 어린이 동화의 캐릭터 역할극을 한다. 반드시 {character_name}으로만 1인칭으로 답한다.

[동화 계획]
{compact_json(plan_context)}

[동화 전문 - {context_source}]
{story_text}

[현재 캐릭터]
{json.dumps(profile, ensure_ascii=False)}

[이전 대화]
{history_text}

[아이 이름] {user_name}
[아이 메시지] {user_message}

규칙:
- 캐릭터는 위 동화에서 실제로 겪은 사건, 관계, 결말을 기억하고 그것과 모순되지 않게 답한다.
- 동화 본문에 없는 일을 사실처럼 꾸며내지 않는다. 모르면 '내가 이야기에서 보지 못한 일이야'라고 솔직히 말한다.
- AI, 모델, 프롬프트, 서버라는 말은 하지 않는다.
- 캐릭터 성격과 동화의 결말을 뒤집지 않는다.
- 어린이가 이해할 쉬운 한국어 2~4문장으로 답한다.
- suggested_replies에는 서로 다른 짧은 다음 질문 세 개를 넣는다.
- JSON 외의 말이나 코드블록은 쓰지 않는다.
{{"reply":"캐릭터 답장","suggested_replies":["질문1","질문2","질문3"]}}'''
    raw = call_llm(prompt, 520, 0.65, 0.92)
    data = extract_json_loose(raw)
    if not isinstance(data, dict):
        data = {}
    reply = clean_model_text(data.get("reply", "")) or clean_model_text(raw)
    suggestions = data.get("suggested_replies") if isinstance(data.get("suggested_replies"), list) else []
    return {
        "reply": reply[:1200],
        "suggested_replies": [clean_model_text(item)[:80] for item in suggestions if clean_model_text(item)][:3],
    }


@api_app.post("/story/image")
async def api_image(request: Request):
    body = await request.json()
    try:
        image = generate_illustration(clean_model_text(body.get("story_text", "")), clean_model_text(body.get("genre", "")) or "판타지")
        buffer = BytesIO()
        image.save(buffer, format="PNG")
        return {"image_b64": base64.b64encode(buffer.getvalue()).decode("ascii")}
    except Exception as error:
        api_error(503, f"삽화 생성 실패: {error}")


@api_app.post("/api/stt/warm-up")
def api_stt_warm_up():
    try:
        ensure_stt_model()
        return {"status": "ready", "engine": f"faster-whisper {STT_MODEL_NAME}", "device": STT_DEVICE}
    except Exception as error:
        api_error(503, f"음성 인식 준비 실패: {error}")


@api_app.post("/api/stt")
async def api_stt(request: Request):
    form = await request.form()
    uploaded = form.get("audio") or form.get("audio_file")
    if uploaded is None or not hasattr(uploaded, "read"):
        api_error(400, "audio WAV 파일을 함께 보내 주세요.")
    audio = await uploaded.read()
    if len(audio) < 800:
        api_error(400, "녹음이 너무 짧습니다.")
    language = clean_model_text(form.get("language", "ko")) or "ko"
    suffix = Path(getattr(uploaded, "filename", "voice.wav")).suffix or ".wav"
    temporary_path = None
    try:
        with NamedTemporaryFile(suffix=suffix, delete=False) as temporary_file:
            temporary_file.write(audio)
            temporary_path = temporary_file.name
        model = ensure_stt_model()
        segments, info = model.transcribe(
            temporary_path,
            language=language,
            task="transcribe",
            beam_size=5,
            vad_filter=True,
            condition_on_previous_text=False,
            initial_prompt="어린이 동화 속 캐릭터에게 하는 한국어 질문입니다.",
        )
        text = clean_model_text(" ".join(segment.text for segment in segments))
        if not text:
            api_error(422, "음성을 이해하지 못했어요. 더 또렷하게 말해 주세요.")
        return {
            "text": text,
            "language": language,
            "duration": round(float(getattr(info, "duration", 0.0)), 2),
            "engine": f"faster-whisper {STT_MODEL_NAME}",
        }
    except HTTPException:
        raise
    except Exception as error:
        api_error(503, f"음성 인식 실패: {error}")
    finally:
        if temporary_path:
            try:
                os.unlink(temporary_path)
            except FileNotFoundError:
                pass


@api_app.post("/api/tts/warm-up")
def api_tts_warm_up():
    try:
        ensure_xtts_sidecar()
        return {"status": "ready", "engine": "XTTS-v2", "device": XTTS_DEVICE}
    except Exception as error:
        api_error(503, f"XTTS 준비 실패: {error}")


@api_app.post("/api/tts")
async def api_tts(request: Request):
    content_type = request.headers.get("content-type", "")
    sample = None
    if content_type.startswith("multipart/"):
        form = await request.form()
        text = clean_model_text(form.get("text", ""))
        uploaded = form.get("speaker_wav")
        if uploaded is not None and hasattr(uploaded, "read"):
            sample = await uploaded.read()
    else:
        body = await request.json()
        text = clean_model_text(body.get("text", ""))
        encoded = body.get("speaker_wav_b64", "")
        if encoded:
            try:
                sample = base64.b64decode(encoded, validate=True)
            except Exception:
                api_error(400, "speaker_wav_b64가 올바른 WAV 데이터가 아닙니다.")
    if not text:
        api_error(400, "읽을 동화 텍스트가 비어 있습니다.")
    if not sample:
        api_error(400, "XTTS-v2는 내 목소리 녹음 WAV가 필요합니다.")
    if sample[:4] != b"RIFF" or sample[8:12] != b"WAVE":
        api_error(400, "PCM WAV 형식의 내 목소리 녹음 파일이 필요합니다.")
    try:
        with TemporaryDirectory() as directory:
            speaker_path = Path(directory) / "speaker.wav"
            speaker_path.write_bytes(sample)
            audio = synthesize_with_my_voice(text, speaker_path)
        return Response(content=audio, media_type="audio/wav")
    except Exception as error:
        api_error(503, f"XTTS 음성 합성 실패: {error}")


if "api_server_thread" not in globals() or not api_server_thread.is_alive():
    nest_asyncio.apply()
    api_config = uvicorn.Config(api_app, host="127.0.0.1", port=8000, log_level="warning")
    api_server = uvicorn.Server(api_config)
    api_server_thread = threading.Thread(target=api_server.run, daemon=True)
    api_server_thread.start()
    time.sleep(2)
    print("FastAPI 서버를 127.0.0.1:8000에서 시작했습니다.")
else:
    print("FastAPI 서버가 이미 실행 중입니다.")


TUNNEL_LOG_PATH = Path("/tmp/test_v1_cloudflared.log")
if "api_tunnel_process" not in globals() or api_tunnel_process.poll() is not None:
    if TUNNEL_LOG_PATH.exists():
        TUNNEL_LOG_PATH.unlink()
    api_tunnel_process = subprocess.Popen(
        [
            "cloudflared", "tunnel", "--url", "http://127.0.0.1:8000",
            "--no-autoupdate", "--logfile", str(TUNNEL_LOG_PATH), "--loglevel", "info",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    public_url = ""
    for _ in range(90):
        time.sleep(1)
        if TUNNEL_LOG_PATH.exists():
            matched = re.search(
                r"https://[-a-z0-9]+\.trycloudflare\.com",
                TUNNEL_LOG_PATH.read_text(errors="replace"),
            )
            if matched:
                public_url = matched.group(0)
                break
    if not public_url:
        details = TUNNEL_LOG_PATH.read_text(errors="replace")[-3000:] if TUNNEL_LOG_PATH.exists() else ""
        raise RuntimeError("Cloudflare 터널 URL을 받지 못했습니다.\n" + details)
    API_PUBLIC_URL = public_url
elif "API_PUBLIC_URL" not in globals():
    raise RuntimeError("이미 실행 중인 터널의 URL을 알 수 없습니다. 런타임을 다시 시작한 뒤 셀 7을 실행하세요.")

print("=" * 68)
print("Flutter .env에 아래 두 줄을 넣으세요.")
print(f"AI_API_BASE_URL={API_PUBLIC_URL}")
print(f"TTS_API_BASE_URL={API_PUBLIC_URL}")
print("=" * 68)


In [ ]:
# 셀 8: 요청 하나로 계획을 만들고, 7번의 무작위 선택을 반영해 8단계 결말까지 생성합니다.
USER_REQUEST = "용에게 납치된 공주님을 구하러 가는 어린 용사의 따뜻한 모험 동화"
STORY_GENRE = "판타지"
RANDOM_CHOICE_SEED = secrets.randbits(64)
rng = random.Random(RANDOM_CHOICE_SEED)

STORY_PLAN = make_story_plan(USER_REQUEST, genre=STORY_GENRE)
print("[제목]", STORY_PLAN["title"])
print("[교훈]", STORY_PLAN["lesson"])
print("[미리 정해진 등장인물/중요 존재]", json.dumps(STORY_PLAN["cast"], ensure_ascii=False, indent=2))
print("[8단계 계획]")
for stage in STORY_PLAN["stages"]:
    print(f"{stage['number']}. {stage['title']} | {stage['goal']}")

game = start_game(STORY_PLAN, STORY_GENRE)
game["settings"]["run_mode"] = "random_demo"
print("\n[무작위 선택 시드]", RANDOM_CHOICE_SEED)

while True:
    scene = game["state"]["scenes"][-1]
    print(f"\n===== {scene['stage']}단계: {scene['title']} =====\n")
    print(scene["text"])
    if game["state"]["ended"]:
        break
    assessment = game["state"].get("choice_assessment", {})
    print("\n[선택지 구분]", assessment.get("status"), assessment.get("reason"))
    print("[참고] 구분이 약해도 진행합니다. 이 단계는 누적 선호 해석에서 제외됩니다."
          if not assessment.get("usable_for_pattern_analysis") else "[참고] 모델의 잠정적인 선택지 구분입니다.")
    print("\n[선택지]")
    for index, choice in enumerate(game["state"]["choices"], 1):
        print(f"{index}. {choice['text']}")
    selected_index = rng.randrange(3)
    selected = game["state"]["choices"][selected_index]
    print(f"\n[무작위 선택] {selected_index + 1}. {selected['text']}")
    game = choose_and_advance(game, selected_index)
    observation = game["state"]["history"][-1]["choice"].get("observation", {})
    print("[선택 행동]", observation.get("behavior_tags", []))

assert len(game["state"]["scenes"]) == 8
assert len(game["state"]["history"]) == 7
print("\n[완료] 7번의 무작위 선택을 기록하고 8단계 결말까지 생성했습니다.")


In [ ]:
# 셀 9: 선택사항 - 각 장면의 삽화를 만듭니다.
# L4에서는 DreamShaper가 CPU로 동작하므로 오래 걸립니다. 필요한 장면만 True로 바꾸세요.
GENERATE_STAGE_ILLUSTRATIONS = False
ILLUSTRATION_STAGES = [1, 4, 7, 8]

if GENERATE_STAGE_ILLUSTRATIONS:
    from IPython.display import display
    illustrations = {}
    for scene in game["state"]["scenes"]:
        if scene["stage"] not in ILLUSTRATION_STAGES:
            continue
        print(f"삽화 생성: {scene['stage']}단계 {scene['title']}")
        image = generate_illustration(scene["text"], STORY_GENRE)
        illustrations[scene["stage"]] = image
        display(image)
else:
    print("삽화 생성은 건너뛰었습니다. GENERATE_STAGE_ILLUSTRATIONS=True로 바꾸면 DreamShaper 8을 사용합니다.")


In [ ]:
# 셀 10: 서버와 동일한 근거·비교 기준으로 엔딩 해설을 만듭니다.
# 셀 8은 무작위 선택 테스트이며 실제 아이의 선호를 측정한 자료가 아닙니다.
def p_psych_report(game):
    selected = [item["choice"] for item in game["state"]["history"]]
    return api_psych_prompt(
        game["plan"]["title"],
        [item["text"] for item in selected],
        [],
        _psych_story_context(game),
    )


def normalize_psych_report(raw, game):
    selected = [item["choice"] for item in game["state"]["history"]]
    return normalize_api_psych(
        raw, [item["text"] for item in selected],
        [],
        context=_psych_story_context(game),
    )


PSYCH_REPORT = normalize_psych_report(
    call_llm(p_psych_report(game), max_tokens=1250, temperature=0.25, top_p=0.85), game
)
print("[분석 유형]", PSYCH_REPORT["type"])
print("[감정 점수]", "생성하지 않음")
print("[비교 가능한 기록]", PSYCH_REPORT["analysis_coverage"])
print("\n" + PSYCH_REPORT["description"])
print("\n[선택별 해설]")
for insight in PSYCH_REPORT["choice_insights"]:
    print("-", insight)
print("\n[함께 나눌 질문]")
for question in PSYCH_REPORT["caregiver_prompts"]:
    print("-", question)
print(PSYCH_REPORT["analysis_caution"])


In [ ]:
# 셀 11: 계획에 미리 정해진 등장인물과 대화합니다.
# 캐릭터는 전체 계획과 완성된 8개 장면을 함께 받아 동화 속 사건을 기억합니다.
def character_profiles(game):
    return [
        {
            "name": item["name"],
            "description": item["description"],
            "desire": item["desire"],
            "enters_at": item["enters_at"],
        }
        for item in game["plan"]["cast"]
    ]


def character_chat(game, character_name, user_message, history=None, user_name="동화 친구"):
    profile = next((item for item in character_profiles(game) if item["name"] == character_name), None)
    if profile is None:
        raise ValueError("계획에 없는 캐릭터입니다. character_profiles(game) 목록에서 골라주세요.")
    history = history or []
    history_text = "\n".join(
        f"{'아이' if item.get('role') == 'user' else character_name}: {str(item.get('content', ''))[:400]}"
        for item in history[-12:] if isinstance(item, dict)
    ) or "아직 대화 없음"
    prompt = f"""너는 완성된 어린이 동화의 캐릭터 역할극을 한다. 아래 프로필의 {character_name}으로만 1인칭 대답을 한다.

[동화 계획]
{compact_json(game['plan'])}

[완성된 동화]
{all_story_text(game)[-11000:]}

[현재 캐릭터]
{json.dumps(profile, ensure_ascii=False)}

[이전 대화]
{history_text}

[아이 이름] {user_name}
[아이 메시지] {user_message}

규칙:
- 캐릭터는 동화에서 실제로 겪은 사건과 관계를 기억한다.
- AI, 모델, 프롬프트, 설정이라는 말은 하지 않는다.
- 캐릭터 성격과 결말을 뒤집지 않는다.
- 어린이가 이해할 쉬운 한국어 2~4문장으로 답한다.
- suggested_replies에는 서로 다른 짧은 다음 질문 세 개를 넣는다.
- JSON 외의 말은 쓰지 않는다.
{{"reply":"캐릭터 답장","suggested_replies":["질문1","질문2","질문3"]}}"""
    raw = call_llm(prompt, max_tokens=520, temperature=0.65, top_p=0.92)
    data = extract_json_loose(raw)
    if not isinstance(data, dict):
        data = {}
    reply = clean_model_text(data.get("reply", "")) or clean_model_text(raw)
    suggestions = data.get("suggested_replies", []) if isinstance(data.get("suggested_replies"), list) else []
    return {
        "character": profile,
        "reply": reply[:1200],
        "suggested_replies": [clean_model_text(item)[:80] for item in suggestions if clean_model_text(item)][:3],
    }


AVAILABLE_CHARACTERS = character_profiles(game)
print("[대화 가능한 미리 설정된 인물]")
for index, profile in enumerate(AVAILABLE_CHARACTERS, 1):
    print(f"{index}. {profile['name']} - {profile['description']}")

# 아래 이름을 위 목록에서 골라 바꾸세요.
CHAT_CHARACTER_NAME = AVAILABLE_CHARACTERS[0]["name"]
CHAT_MESSAGE = "모험을 하면서 가장 기억에 남는 순간은 무엇이었어?"
CHAT_RESULT = character_chat(game, CHAT_CHARACTER_NAME, CHAT_MESSAGE)
print(f"\n[{CHAT_CHARACTER_NAME}] {CHAT_RESULT['reply']}")
print("[다음 질문]", CHAT_RESULT["suggested_replies"])


In [ ]:
# 셀 12: 선택사항 - 내 목소리 WAV로 XTTS-v2 낭독을 테스트합니다.
# Flutter에서 녹음해 받은 PCM WAV 경로를 넣으면 같은 방식으로 합성됩니다.
SPEAKER_WAV_PATH = ""  # 예: /content/my_voice.wav
TTS_TEXT = game["state"]["scenes"][0]["text"]

if SPEAKER_WAV_PATH:
    from IPython.display import Audio, display
    audio_bytes = synthesize_with_my_voice(TTS_TEXT, SPEAKER_WAV_PATH)
    output_wav = Path("/content/test_v1_xtts_output.wav")
    output_wav.write_bytes(audio_bytes)
    print("XTTS 합성 완료:", output_wav)
    display(Audio(data=audio_bytes, rate=24000, autoplay=False))
else:
    print("SPEAKER_WAV_PATH가 비어 있어 XTTS 모델 로드를 건너뛰었습니다.")
    print("Flutter 녹음 파일 또는 3초 이상 PCM WAV의 경로를 넣으면 XTTS-v2로 내 목소리 낭독을 테스트합니다.")


In [ ]:
# 셀 13: 전체 동화 JSON과 사람의 검토가 필요한 선택 분석 초안 JSONL을 저장합니다.
from datetime import datetime

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = Path("/content") / f"test_v1_story_{stamp}.json"
output_path.write_text(json.dumps({
    "schema_version": 2,
    "record_id": game.get("record_id"),
    "run_mode": game.get("settings", {}).get("run_mode", "unknown"),
    "plan": game["plan"],
    "scenes": game["state"]["scenes"],
    "selected_choices": game["state"]["history"],
    "psych_report": globals().get("PSYCH_REPORT"),
    "ended": game["state"]["ended"],
}, ensure_ascii=False, indent=2), encoding="utf-8")
decision_path = Path("/content") / f"test_v1_decisions_{stamp}.jsonl"
examples = decision_training_examples(game)
decision_path.write_text(
    "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in examples),
    encoding="utf-8",
)
print("전체 동화:", output_path)
print("선택 분석 검토용 자료:", decision_path, f"({len(examples)}건)")
print("모델 라벨은 정답이 아닙니다. 사람 검토 후 학습에 쓰고, 같은 story_id는 같은 분할에 두세요.")
print("random_demo는 실제 사용자 선택 데이터와 구분해야 합니다.")

# 필요하면 아래 주석을 풀어 내려받습니다.
# from google.colab import files
# files.download(str(output_path))
# files.download(str(decision_path))
